In [16]:
import pandas as pd
import glob

# Reference file
brain_file = "20250830_GeneId_Brain_50PerRegion-Unique.tsv"

# Load brain genes (first column, skip header)
brain = pd.read_csv(brain_file, sep="\t", usecols=[0], skiprows=1, header=None)[0]
brain_set = set(brain)

# Process each TSP file
for tsp_file in glob.glob("20250413_COO_MuSiC-Decon/TSP*filtered.txt"):
    tsp = pd.read_csv(tsp_file, sep="\t", usecols=[0], skiprows=1, header=None)[0]
    tsp_set = set(tsp)

    matches = len(tsp_set & brain_set)
    only_in_tsp = len(tsp_set - brain_set)
    only_in_brain = len(brain_set - tsp_set)

    print(f"File: {tsp_file}")
    print(f"  Matches: {matches}")
    print(f"  Only in TSP: {only_in_tsp}")
    print(f"  Only in Brain: {only_in_brain}")
    print(f"  TSP total: {len(tsp_set)}, Brain total: {len(brain_set)}\n")


File: 20250413_COO_MuSiC-Decon/TSP-HBA_Inner_100each_seed42_filtered.txt
  Matches: 26627
  Only in TSP: 18480
  Only in Brain: 27965
  TSP total: 45107, Brain total: 54592

File: 20250413_COO_MuSiC-Decon/TSP-BDa_Inner_100each_seed42_filtered.txt
  Matches: 17960
  Only in TSP: 173
  Only in Brain: 36632
  TSP total: 18133, Brain total: 54592

File: 20250413_COO_MuSiC-Decon/TSP-BDa_Outer_100each_seed42_filtered.txt
  Matches: 28193
  Only in TSP: 22501
  Only in Brain: 26399
  TSP total: 50694, Brain total: 54592



Commands to re-name the reference data (GTEx) to match the namings of genes as of gencode v46.

Done for:
- All GTEx Brain Per Region data

The input refers to the contents of the 20250405_TOO-Matrices file. Duplicate names are not allowed so are removed to keep the occurence with the highest counts.

In [17]:
import pandas as pd
import re
import glob

# === Input files ===
brain_file = "20250830_GeneId_Brain_50PerRegion-Unique.tsv"
tsp_files = glob.glob("20250413_COO_MuSiC-Decon/TSP*filtered.txt")
ref_path   = "20250830_Both_Brain_50PerRegion-Unique.tsv"
gtf_path   = "gencode.v46.annotation.gtf"

# --- Load Brain file ---
brain = pd.read_csv(brain_file, sep="\t")
brain_genes = set(brain.iloc[:, 0])

# --- Load TSP genes ---
tsp_genes_all = set()
for f in tsp_files:
    tsp_genes = pd.read_csv(f, sep="\t", usecols=[0], skiprows=1, header=None)[0]
    tsp_genes_all.update(tsp_genes)

# Genes to remap = only in Brain
only_in_brain = brain_genes - tsp_genes_all
print(f"Brain genes total: {len(brain_genes)}")
print(f"Only in Brain (to remap): {len(only_in_brain)}")

# --- Load reference map (GeneName -> GeneID_clean) ---
reference = pd.read_csv(ref_path, sep="\t", index_col=False)
reference.columns.values[1] = "GeneName"
reference = reference.iloc[:, :2]
reference["GeneID_clean"] = reference["GeneID"].astype(str).str.split(".").str[0]
ref_map = dict(zip(reference["GeneName"], reference["GeneID_clean"]))

# --- Load GTF (GeneID_clean -> GeneName) ---
def extract_gtf_info(attr, key):
    match = re.search(f'{key} "([^"]+)"', attr)
    return match.group(1) if match else None

gtf = pd.read_csv(gtf_path, sep="\t", comment="#", header=None)
gtf.columns = ["seqname", "source", "feature", "start", "end", "score", "strand", "frame", "attribute"]
genes_gtf = gtf[gtf["feature"] == "gene"].copy()
genes_gtf["GeneID"] = genes_gtf["attribute"].apply(lambda x: extract_gtf_info(x, "gene_id"))
genes_gtf["GeneID_clean"] = genes_gtf["GeneID"].str.split(".").str[0]
genes_gtf["GeneName"] = genes_gtf["attribute"].apply(lambda x: extract_gtf_info(x, "gene_name"))
gtf_map = dict(zip(genes_gtf["GeneID_clean"], genes_gtf["GeneName"]))

# --- Remap only the "Only in Brain" genes ---
remap_dict = {}
for g in only_in_brain:
    if g in ref_map:
        gid = ref_map[g]
        if gid in gtf_map:
            remap_dict[g] = gtf_map[gid]

print(f"Remapping {len(remap_dict)} of {len(only_in_brain)} Only-in-Brain genes")

# --- Apply renaming only to those genes ---
brain.iloc[:, 0] = brain.iloc[:, 0].apply(lambda x: remap_dict.get(x, x))

# --- Handle duplicates if remapping introduces collisions ---
if brain.iloc[:, 0].duplicated().any():
    print("Duplicate gene names after remapping — keeping row with highest row sum")
    brain["row_sum"] = brain.iloc[:, 1:].sum(axis=1)
    brain = brain.sort_values("row_sum", ascending=False).drop_duplicates(subset=brain.columns[0], keep="first")
    brain = brain.drop(columns="row_sum")

# --- Save new Brain file ---
out_path = brain_file.replace(".tsv", ".OnlyInBrainRemapped.txt")
brain.to_csv(out_path, sep="\t", index=False)
print(f"Saved updated Brain file → {out_path}")


Brain genes total: 54592
Only in Brain (to remap): 25530


/tmp/ipykernel_1013056/446679427.py:27: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  reference = pd.read_csv(ref_path, sep="\t", index_col=False)


Remapping 24834 of 25530 Only-in-Brain genes
Duplicate gene names after remapping — keeping row with highest row sum
Saved updated Brain file → 20250830_GeneId_Brain_50PerRegion-Unique.OnlyInBrainRemapped.txt


In [18]:
import pandas as pd
import glob

# Reference file
brain_file = "20250830_GeneId_Brain_50PerRegion-Unique.OnlyInBrainRemapped.txt"

# Load brain genes (first column, skip header)
brain = pd.read_csv(brain_file, sep="\t", usecols=[0], skiprows=1, header=None)[0]
brain_set = set(brain)

# Process each TSP file
for tsp_file in glob.glob("20250413_COO_MuSiC-Decon/TSP*filtered.txt"):
    tsp = pd.read_csv(tsp_file, sep="\t", usecols=[0], skiprows=1, header=None)[0]
    tsp_set = set(tsp)

    matches = len(tsp_set & brain_set)
    only_in_tsp = len(tsp_set - brain_set)
    only_in_brain = len(brain_set - tsp_set)

    print(f"File: {tsp_file}")
    print(f"  Matches: {matches}")
    print(f"  Only in TSP: {only_in_tsp}")
    print(f"  Only in Brain: {only_in_brain}")
    print(f"  TSP total: {len(tsp_set)}, Brain total: {len(brain_set)}\n")


File: 20250413_COO_MuSiC-Decon/TSP-HBA_Inner_100each_seed42_filtered.txt
  Matches: 39841
  Only in TSP: 5266
  Only in Brain: 14714
  TSP total: 45107, Brain total: 54555

File: 20250413_COO_MuSiC-Decon/TSP-BDa_Inner_100each_seed42_filtered.txt
  Matches: 17972
  Only in TSP: 161
  Only in Brain: 36583
  TSP total: 18133, Brain total: 54555

File: 20250413_COO_MuSiC-Decon/TSP-BDa_Outer_100each_seed42_filtered.txt
  Matches: 41296
  Only in TSP: 9398
  Only in Brain: 13259
  TSP total: 50694, Brain total: 54555

